# Neural reduced GP

AutoEncoder / VAE / supervised AutoEncoder を軽量設定で構築し、非線形 latent representation の使い方を確認します。


## データ


In [ ]:
import torch
from robotorchan.models import AutoEncoderGP, VAEGP, SupervisedAutoEncoderGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.rand(20, 6)
Y = torch.sin(4 * X[:, :1]) + X[:, 1:2]


## 軽量なreducer学習


In [ ]:
models = {
    "AE": AutoEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
    "VAE": VAEGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
    "SupervisedAE": SupervisedAutoEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
}
for name, model in models.items():
    print(name, model.reduced_input_dim)


## Posterior


In [ ]:
X_test = torch.rand(2, 6)
for name, model in models.items():
    print(name, model.posterior(X_test).mean.squeeze(-1))


## Model fitting

Reducerは構築時に学習済みですが、GP部分は既存の`make_mll()`で別途学習します。


In [ ]:
from botorch.fit import fit_gpytorch_mll

for name, model in models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    print(name, "GP fitted")


## Acquisition-based candidate generation and evaluation

各モデルについて元の6次元空間でqLogEIを最適化し、提案点を目的関数で評価します。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler


def objective(X):
    return torch.sin(4 * X[..., :1]) + X[..., 1:2]


bounds = torch.stack([torch.zeros(6), torch.ones(6)])
for name, model in models.items():
    acqf = qLogExpectedImprovement(
        model=model,
        best_f=Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=2,
        raw_samples=32,
    )
    assert candidate.shape == torch.Size([1, 6])
    print(name, "candidate:", candidate)
    print(name, "acquisition:", value)
    print(name, "objective:", objective(candidate))
